# Chapter 13 — Pre-Training Demo## Next-Token Prediction at Scale[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**No GPU required. Runtime: ~15 minutes.**

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
print('Ready')

## 13.1 The Pre-Training Objective: Cross-Entropy on Next-Token Prediction

In [ ]:
# Demonstrate the pre-training loss on a small vocabulary
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'a', '<EOS>']
vocab_size = len(vocab)
w2i = {w: i for i, w in enumerate(vocab)}

sentence = ['the', 'cat', 'sat', 'on', 'the', 'mat']
print('Training sentence:', sentence)
print()
print('Next-token prediction training pairs:')
for i in range(len(sentence)-1):
    context = sentence[:i+1]
    target  = sentence[i+1]
    print(f'  Context: {context} → Target: "{target}"')

## 13.2 What SFT AddsSFT applies the same cross-entropy loss but only on the **response** tokens, not the prompt tokens.

In [ ]:
# Illustrate the SFT token masking
prompt_text = 'What is the capital of France?'
response_text = 'The capital of France is Paris.'

print('SFT Training Objective:')
print(f'  Full sequence: [PROMPT] {prompt_text} [RESPONSE] {response_text}')
print()
print('  Loss is computed ONLY on response tokens:')
response_words = response_text.split()
for i, word in enumerate(response_words):
    context = prompt_text + ' ' + ' '.join(response_words[:i])
    print(f'    Context: ...{context[-40:]} → Predict: "{word}"')
print()
print('  Prompt tokens contribute to the context but NOT to the loss gradient.')
print('  This is implemented via the labels=-100 masking in HuggingFace transformers.')

## ✅ Key Takeaways1. Pre-training = next-token prediction on raw text (self-supervised, no labels)2. SFT = same objective but only on response tokens, with prompt as masked context3. RLHF adds a reward signal on top of SFT to align with human preferences4. The computational graph is identical for all three — only the loss signal changes